# Garmin Coach - AI Chat with Your Training Data

This notebook lets you chat with an LLM about your Garmin workout data stored in CDF. Uses `cognite-ai` SmartDataframe for natural language queries.

**Prerequisites:**
- Run the `garmin_activities_extractor` function at least once to ingest data
- Set up `.env` with CDF credentials (or use default CogniteClient)

In [ ]:
import pandas as pd
from cognite.client import CogniteClient
from cognite.client.data_classes.data_modeling import ViewId

# Use get_client if you have .env in project root, or CogniteClient() for default config
import os
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
try:
    from get_client import get_client
    env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
    client = get_client(env_path)
except Exception:
    client = CogniteClient()

In [ ]:
# Load workout data from CDF data model
SPACE_NAME = "garmin_coach_space"
workout_view = ViewId(SPACE_NAME, "Workout", "1.0")

workouts = client.data_modeling.instances.list(
    sources=workout_view,
    space=SPACE_NAME,
    limit=-1,
)

# Convert to pandas DataFrame
df = workouts.to_pandas(convert_timestamps=True)
print(f"Loaded {len(df)} workouts")
df.head()

## Chat with SmartDataframe

Use natural language to ask questions about your training data. Examples:
- "Based on my last 4 weeks of running, is my average heart rate trending up or down at 5:00/km pace?"
- "What's my total distance this month?"
- "Compare my running pace in January vs February"

In [ ]:
# Load cognite-ai (async - run this cell first)
from cognite.ai import load_pandasai

SmartDataframe, SmartDatalake, Agent = await load_pandasai()

In [ ]:
# Create SmartDataframe and chat
smart_df = SmartDataframe(df, cognite_client=client)

# Example query - change this to your question
smart_df.chat("Summarize my workout data: total runs, total distance, and average heart rate by activity type.")

In [ ]:
# Ask follow-up questions
smart_df.chat("What is my average pace for running activities in the last 30 days?")

## Alternative: Use Events

If the Data Model has no data yet, you can load from Events (workout summaries) instead.

In [ ]:
# Load from Events as fallback (when Data Model is empty)
events = client.events.list(
    external_id_prefix="garmin_activity_",
    limit=500,
)

events_df = pd.DataFrame([
    {
        "external_id": e.external_id,
        "start_time": e.start_time,
        "end_time": e.end_time,
        **e.metadata or {},
    }
    for e in events
])
events_df